# LMCompress — Colab H100 runbook

Reproduction of **Li et al., *Lossless data compression by large models*, Nature Machine Intelligence 7:794–799 (2025)**, capsule 4957991.

This notebook does the two jobs that exceed the local machine's budget:

| Part | Job | Why Colab | Est. H100 time |
|---|---|---|---|
| **A** | Full text evaluation at the paper's settings (8-bit base, ctx 8192) | ~8 h on the local RTX 5070 | 20–40 min |
| **B** | LoRA fine-tuning of the text modality, paper recipe | days locally (4-bit + offload) | 1.5–3 h |
| **C** | **Training-quantity ablation** — the experiment the paper never ran | — | 1–2 h |

## What is already established locally (do not redo)

| Modality | Result | Paper |
|---|---|---|
| text, raw Llama-3-8B | **8.86 %** | 9.08 % |
| text, + shipped LoRA | **9.12 %** | 9.06 % |
| audio baselines | OptimFROG 27.31 / WavPack 35.30 / FLAC 45.33 % | 25.42 / 33.15 / 41.26 % |
| text baselines | brotli 28.81 / zstd 31.99 / 7z-equiv 31.73 % | 27.93 / 30.45 / 30.25 % |

So the port is validated. What Colab adds is **scale** and the **fine-tuning ablation**.

## Two warnings about the reference implementation

1. Its text path loads Llama-3-8B **in float32 with no `device_map`** (`from_pretrained(model_path)` then `.to('cuda')`), i.e. ~32 GB. This notebook uses 8-bit (paper's setting) or 4-bit with `device_map`.
2. Its audio path's 4.13× is **not** a next-byte measurement — the bundled WAV's order-1 entropy is 3.17 bits/byte, below the 1.94 it reports. This notebook does not attempt to reproduce that number; see `docs/REPRODUCTION_NOTES.md` §3.3.


## 0. Environment

Installs pinned versions so the LoRA adapter in the capsule loads. `peft` must be recent enough for `transformers` 5.x.


In [ ]:
import subprocess, sys

PKGS = [
    'torch>=2.4',
    'transformers>=4.44',
    'peft>=0.13',
    'accelerate>=1.0',
    'bitsandbytes>=0.45',
    'datasets',
    'pandas',
    'zstandard',
    'brotli',
    'sentencepiece',
    'tqdm',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PKGS], check=True)

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print('GPU', p.name, f'{p.total_memory/1e9:.1f} GB')
else:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> H100 / A100.')


## 1. Inputs

Three ways to supply the data; pick one.

* **Google Drive** (recommended for a multi-hour job: survives a runtime reset and lets checkpoints persist) — set `USE_DRIVE = True`.
* **Kaggle / HF datasets** — set `FINE_TUNE_CORPUS` and let the download cell fetch it.
* **Upload a zip** — the same archive the local `tools/pack_colab.sh` produces.

The corpus layout expected throughout is single-column CSV with a `TEXT` column, exactly what the paper's `preprocess_text_dataset.py` writes.


In [ ]:
import os
from pathlib import Path

USE_DRIVE = False          # set True to persist checkpoints + results to Drive
WORK = Path('/content/lmcompress')
WORK.mkdir(parents=True, exist_ok=True)

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = Path('/content/drive/MyDrive/lmcompress')
    WORK.mkdir(parents=True, exist_ok=True)

# Corpus: 64 MB train / 16 MB validation / rest test, per the paper.
DATA = WORK / 'data'
(DATA / 'train').mkdir(parents=True, exist_ok=True)
(DATA / 'val').mkdir(parents=True, exist_ok=True)
(DATA / 'test').mkdir(parents=True, exist_ok=True)
CKPT = WORK / 'checkpoints'; CKPT.mkdir(exist_ok=True)
RESULTS = WORK / 'results'; RESULTS.mkdir(exist_ok=True)
print('WORK =', WORK)


### 1b. Get the corpus

The paper uses MeDAL (medical) and the eurlex split of Pile of Law (legal). The domain matters: the paper's own numbers show a large fine-tuning gain on eurlex and almost none on MeDAL, and §C below tests exactly that.


In [ ]:
DATASET = 'medal'   # 'medal' | 'eurlex'
TRAIN_MB, VAL_MB = 64, 16

def load_corpus(dataset: str):
    """Return (train_text, val_text, test_text) sized per the paper."""
    if dataset == 'eurlex':
        from datasets import load_dataset
        ds = load_dataset('pile-of-law/pile-of-law', 'eurlex', split='train',
                          streaming=True)
        texts = []
        total = 0
        for row in ds:
            t = row.get('text') or ''
            texts.append(t); total += len(t.encode())
            if total > (TRAIN_MB + VAL_MB + 32) * 1024 * 1024:
                break
        blob = '\n'.join(texts)
    else:
        # MeDAL: https://github.com/McGill-NLP/medal
        from datasets import load_dataset
        ds = load_dataset('McGill-NLP/medal', split='train', streaming=True)
        texts, total = [], 0
        for row in ds:
            t = row.get('abstract_text') or row.get('TEXT') or ''
            texts.append(t); total += len(t.encode())
            if total > (TRAIN_MB + VAL_MB + 32) * 1024 * 1024:
                break
        blob = '\n'.join(texts)

    b = blob.encode('utf-8')
    n_train = TRAIN_MB * 1024 * 1024
    n_val = VAL_MB * 1024 * 1024
    return (b[:n_train].decode('utf-8', 'ignore'),
            b[n_train:n_train + n_val].decode('utf-8', 'ignore'),
            b[n_train + n_val:].decode('utf-8', 'ignore'))

train_txt, val_txt, test_txt = load_corpus(DATASET)
for name, t in (('train', train_txt), ('val', val_txt), ('test', test_txt)):
    print(f'{name:5s} {len(t.encode())/1e6:8.2f} MB  {len(t):>10} chars')


In [ ]:
# Write the corpus out as single-column TEXT CSVs, chunked so that a
# single test file is a manageable unit of work (the paper chunks at 100 KB).
import pandas as pd

CHUNK = 100 * 1024
def write_csvs(text: str, outdir: Path, prefix: str):
    outdir.mkdir(parents=True, exist_ok=True)
    i, buf, size = 0, [], 0
    for line in text.split('\n'):
        buf.append(line); size += len(line.encode()) + 1
        if size >= CHUNK:
            pd.DataFrame({'TEXT': ['\n'.join(buf)]}).to_csv(
                outdir / f'{prefix}_{i}.csv', index=False)
            i += 1; buf, size = [], 0
    if buf:
        pd.DataFrame({'TEXT': ['\n'.join(buf)]}).to_csv(
            outdir / f'{prefix}_{i}.csv', index=False)
        i += 1
    return i

n_tr = write_csvs(train_txt, DATA / 'train', 'train')
n_va = write_csvs(val_txt, DATA / 'val', 'val')
n_te = write_csvs(test_txt, DATA / 'test', 'test')
print(f'train {n_tr} chunks, val {n_va}, test {n_te}')


## 2. The compression engine

Same design as the local package: an arithmetic coder driven by the model's own per-token probability, with the reported code length being the exact cross-entropy `-log2 p(true token)`. Two properties matter:

* **it is lossless** — the coder quantises every symbol to a non-empty interval, so even a symbol the model rates at ~0 is codable and decodable (a minimum-interval-width rule; without it the coder silently corrupts the stream on rare symbols, which is a bug this port hit and fixed);
* **it is checkpointed per window**, so a Colab disconnect costs one forward pass, not the whole run.


In [ ]:
import json, math, os, time
from dataclasses import dataclass, field
from typing import Iterator, List, Optional, Sequence
import numpy as np
import torch

PRECISION = 32
FULL = 1 << PRECISION
MASK = FULL - 1
HALF, QUARTER, THREE_Q = 1 << 31, 1 << 30, 3 << 30


class ArithmeticEncoder:
    """Range encoder, one bit at a time."""

    def __init__(self, out):
        self.out = out; self.low = 0; self.high = MASK; self.pending = 0

    def _emit(self, bit):
        self.out(bit)

    def _emit_pending(self, bit):
        self._emit(bit)
        for _ in range(self.pending):
            self._emit(bit ^ 1)
        self.pending = 0

    def encode(self, cdf, symbol):
        lo, hi = int(cdf[symbol]), int(cdf[symbol + 1])
        width = self.high - self.low + 1
        new_high = self.low + (width * hi // FULL) - 1
        new_low = self.low + (width * lo // FULL)
        if new_high < new_low:
            raise AssertionError('coder interval collapsed; raise min_width')
        self.high, self.low = new_high, new_low
        while True:
            if self.high < HALF:
                self._emit_pending(0)
            elif self.low >= HALF:
                self._emit_pending(1); self.low -= HALF; self.high -= HALF
            elif self.low >= QUARTER and self.high < THREE_Q:
                self.pending += 1; self.low -= QUARTER; self.high -= QUARTER
            else:
                break
            self.low = (self.low << 1) & MASK
            self.high = ((self.high << 1) | 1) & MASK

    def terminate(self):
        self.pending += 1
        self._emit_pending(0 if self.low < QUARTER else 1)


def probs_to_cdf(probs: Sequence[float], n_symbols: int) -> List[int]:
    """Integer cdf with a guaranteed minimum width per symbol.

    The minimum width is what keeps the coder exact: if a symbol's mass
    rounds to zero while the interval is already narrow, subdividing it
    drives high below low and the decoder returns the wrong symbol.
    """
    w_min = max(1, FULL >> 20)
    if n_symbols * w_min > FULL // 2:
        w_min = max(1, FULL // (2 * n_symbols))
    budget = FULL - n_symbols * w_min
    total = float(sum(probs)) or 1.0
    cdf = [0] * (n_symbols + 1)
    running, prev = 0.0, 0
    for i in range(n_symbols):
        running += max(0.0, float(probs[i]))
        raw = int(running / total * budget) + (i + 1) * w_min
        raw = max(raw, prev + w_min)
        raw = min(raw, FULL - (n_symbols - 1 - i) * w_min)
        cdf[i + 1] = raw; prev = raw
    cdf[n_symbols] = FULL
    return cdf


def sparse_true_symbol_cdf(p_true: float, n_symbols: int, symbol: int,
                           tail_mass: float = 2.0 ** -30) -> List[int]:
    """CDF when only p(true symbol) is known (Llama's 128 k vocab).

    Reconstructing the full posterior per token is unnecessary: the coded
    length is set by p_true / (p_true + tail), so a tail far below the
    model's own floor is invisible.  With 2**-30 the error in the
    reported length is < 1e-6 bits whenever p_true > 1e-6.
    """
    n = int(n_symbols)
    tail_mass = float(min(max(tail_mass, 1e-12), 0.5))
    n_tail = n - 1
    p_true = float(p_true)
    if not p_true > 0 or not math.isfinite(p_true):
        p_true = 1e-12
    p_true = min(p_true, 1.0)
    if p_true > 1.0 - tail_mass:
        tail_mass = max(1e-12, (1.0 - p_true) * 0.5)
    tail_units = min(int(tail_mass * FULL) + n_tail, FULL - n_tail)
    true_units = FULL - tail_units
    per, extra = tail_units // n_tail, tail_units - (tail_units // n_tail) * n_tail
    cdf = [0] * (n + 1)
    acc = 0
    for i in range(n):
        if i == symbol:
            acc += true_units
        else:
            j = i if i < symbol else i - 1
            acc += per + (1 if j < extra else 0)
        cdf[i + 1] = acc
    cdf[n] = FULL
    return cdf


### 2b. The text predictor

Rolling window with per-window checkpointing. Note the alignment bookkeeping: a window of *n* scored tokens is fed *n+1* tokens, the first prediction is skipped (already coded by the previous window), and BOS costs one further position on the first window. Getting this wrong by one shifts every score by one token — which is precisely the defect that makes the reference implementation's audio results uninterpretable.


In [ ]:
class TextPredictor:
    """Codes a token sequence with Llama (+ optional LoRA adapter)."""

    def __init__(self, model, tokenizer, context_window=8192):
        self.model, self.tok = model, tokenizer
        self.ctx = int(context_window)
        self.n_symbols = int(model.config.vocab_size)
        self.bos = getattr(tokenizer, 'bos_token_id', None)
        if self.bos is None:
            self.bos = tokenizer.convert_tokens_to_ids('<|begin_of_text|>')
        self._doc: List[int] = []

    @classmethod
    def load(cls, model_path, lora_path=None, load_in_8bit=False,
             load_in_4bit=False, context_window=8192):
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        tok = AutoTokenizer.from_pretrained(model_path)
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        kw = dict(device_map={'': 0})
        if load_in_8bit:
            kw['quantization_config'] = BitsAndBytesConfig(load_in_8bit=True)
        elif load_in_4bit:
            kw['quantization_config'] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type='nf4',
                bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
        else:
            kw['dtype'] = torch.bfloat16
        model = AutoModelForCausalLM.from_pretrained(model_path, **kw)
        if lora_path:
            from peft import PeftModel
            model = PeftModel.from_pretrained(model, lora_path, is_trainable=False)
            model = model.merge_and_unload()
        model.eval()
        return cls(model, tok, context_window)

    def encode(self, text):
        return self.tok(text, add_special_tokens=False)['input_ids']

    def set_document(self, tokens):
        self._doc = list(tokens)

    def budget(self):
        b = 1 if self.bos is not None else 0
        return max(1, self.ctx - 2 - b)

    def iter_steps(self, start, stop):
        """Yield (p_true, symbol) for document tokens [start, stop)."""
        n_target = stop - start
        if n_target <= 0:
            return
        feed = self._doc[start : min(stop + 1, len(self._doc))]
        n_target = min(n_target, max(0, len(feed) - 1))
        if n_target <= 0:
            return
        history = max(0, self.ctx - (n_target + 1))
        pfx_start = max(0, start - history)
        prefix = self._doc[pfx_start:start]
        bos = [self.bos] if (self.bos is not None and start == 0) else []
        ids = torch.tensor([bos + prefix + feed], dtype=torch.long, device='cuda')
        with torch.no_grad():
            logits = self.model(ids).logits
        off = len(bos) + len(prefix)
        scope = logits[:, off : off + n_target, :].float()
        tgt = ids[:, off + 1 : off + 1 + n_target]
        lp = torch.log_softmax(scope, dim=-1)
        p = lp.gather(-1, tgt.unsqueeze(-1)).squeeze(-1).exp().reshape(-1).cpu().numpy()
        del logits, scope, lp
        for i in range(n_target):
            sym = int(self._doc[start + 1 + i])
            yield float(p[i]), sym


def measure_document(predictor, tokens, cursor_path, unit='doc'):
    """Code `tokens`, checkpointing after every window.  Resumable."""
    budget = predictor.budget()
    windows, s = [], 0
    while s < len(tokens):
        e = min(s + budget, len(tokens))
        windows.append((s, e)); s = e

    state = {'done': 0, 'bits': 0.0, 'symbols': 0}
    if os.path.exists(cursor_path):
        try:
            state.update(json.load(open(cursor_path)))
        except Exception:
            pass

    t0 = time.time()
    for idx, (s, e) in enumerate(windows):
        if idx < state['done']:
            continue
        acc_bits, n_sym = 0.0, 0
        for p_true, sym in predictor.iter_steps(s, e):
            acc_bits += -math.log2(max(p_true, 1e-30)); n_sym += 1
        state['bits'] += acc_bits; state['symbols'] += n_sym
        state['done'] = idx + 1
        json.dump(state, open(cursor_path, 'w'))
    return {'unit': unit, 'tokens': len(tokens), 'symbols': state['symbols'],
            'coded_bits': state['bits'], 'seconds': time.time() - t0,
            'windows': len(windows)}


## Part A — full text evaluation at the paper's settings

The paper's Table 6 uses context length **8192** and an **8-bit** base model. The local run used 4-bit and ctx 4096 to fit a 12 GB card; Colab removes both constraints, so this is the faithful measurement.

Evaluation is on the held-out test split (everything after the first 80 MB), which is what the paper does: *first 64 MB train / next 16 MB validation / rest testing*.


In [ ]:
MODEL_ID = 'meta-llama/Meta-Llama-3-8B'   # needs a HF token with Llama-3 access
HF_TOKEN = ''                              # or use huggingface-cli login
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

predictor = TextPredictor.load(MODEL_ID, load_in_8bit=True, context_window=8192)
print('vocab', predictor.n_symbols, '| window budget', predictor.budget())


In [ ]:
# Ablation switch from the paper's own Supplementary Table 9:
# text CR at ctx 8192 / 6144 / 4096 on MeDAL was 9.06 / 9.16 / 9.36 %.
CTX_ABLATION = [8192, 4096]      # set to [] to skip
N_TEST_FILES = 20                # paper uses all of it; 20 is plenty to compare

import pandas as pd, glob, json

def run_eval(lora_path, tag, ctx_list=(8192,), n_files=N_TEST_FILES):
    files = sorted(glob.glob(str(DATA / 'test' / '*.csv')))[:n_files]
    out_all = {}
    for ctx in ctx_list:
        pred = TextPredictor.load(MODEL_ID, lora_path=lora_path,
                                  load_in_8bit=True, context_window=ctx)
        tot_bits = tot_bytes = 0
        for f in files:
            text = ''.join(str(x) for x in pd.read_csv(f)['TEXT'].tolist())
            raw = text.encode('utf-8')
            toks = pred.encode(text)
            pred.set_document(toks)
            name = f'{tag}_ctx{ctx}_{os.path.basename(f)}'
            res = measure_document(pred, toks, RESULTS / f'cursor_{name}.json', name)
            tot_bits += res['coded_bits']; tot_bytes += len(raw)
        rate = 100.0 * (tot_bits / 8) / tot_bytes
        out_all[ctx] = rate
        print(f'{tag} ctx={ctx}: rate={rate:.4f}%  ratio={100/rate:.3f}x  '
              f'({len(files)} files, {tot_bytes/1e6:.2f} MB)')
        del pred
        torch.cuda.empty_cache()
    return out_all

raw_results = run_eval(None, 'raw_llama', CTX_ABLATION)
json.dump(raw_results, open(RESULTS / 'partA_raw_llama.json', 'w'), indent=2)


### Part A2 — evaluate the adapter that ships with the capsule

The capsule's adapter is the *MeDAL* one (10 epochs, 5 120 steps). Its own `trainer_state.json` shows best eval loss at **step 1 020 (epoch ≈2)**, rising from 2.4708 to 2.6728 by step 5 120 — i.e. the shipped checkpoint is the most overfitted one. Part C tests whether that is why it does not help.


In [ ]:
SHELL_ADAPTER = WORK / 'Llama-3-8B_lora'   # upload the capsule's adapter here
if SHELL_ADAPTER.exists():
    lora_results = run_eval(str(SHELL_ADAPTER), 'shipped_lora', CTX_ABLATION)
    json.dump(lora_results, open(RESULTS / 'partA_shipped_lora.json', 'w'), indent=2)
else:
    print(f'No adapter at {SHELL_ADAPTER}; skipping. '
          'Upload Llama-3-8B_lora/ from the capsule to include it.')


## Part B — reproduce the paper's fine-tuning recipe

Verbatim from Supplementary §S1.4 plus the capsule's `trainer_state.json` and `adapter_config.json`:

| Setting | Value | Source |
|---|---|---|
| LoRA rank / alpha / dropout | 8 / 32 / 0.05 | suppl + adapter_config |
| target modules | q,k,v,o,gate,up,down_proj | adapter_config |
| base precision | 8-bit | suppl |
| train precision | bfloat16 | suppl |
| batch size / grad accum | 1 / 8 | suppl |
| epochs | 10 | suppl |
| LR / schedule | 1e-4 / linear | suppl + trainer_state |
| warmup | 400 steps (suppl says 10 %; the actual run used 7.8 %) | trainer_state |
| optimizer | adamw_torch, wd 0 | trainer_state |

The paper's own run used DeepSpeed ZeRO-2 over 8 pipelines; we use plain `Trainer` on one H100, which changes speed but not the recipe.


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (AutoModelForCausalLM, AutoTokenizer, Trainer,
                          TrainingArguments, BitsAndBytesConfig, default_data_collator)

LORA = dict(r=8, lora_alpha=32, lora_dropout=0.05, bias='none',
            task_type='CAUSAL_LM',
            target_modules=['q_proj','k_proj','v_proj','o_proj',
                            'gate_proj','up_proj','down_proj'])

def build_trainable(max_len=2048):
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=BitsAndBytesConfig(load_in_8bit=True),
        device_map={'': 0})
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, LoraConfig(**LORA))
    model.print_trainable_parameters()   # expect ~21.0 M, i.e. 0.26 %
    return model, tok


class TokenBlocks(torch.utils.data.Dataset):
    """Plain causal-LM blocks: the loss is the same log-loss the coder uses."""

    def __init__(self, text, tok, max_len):
        ids = tok(text, add_special_tokens=False)['input_ids']
        self.blocks = [ids[i:i + max_len] for i in range(0, len(ids) - 1, max_len)]
        self.tok = tok
        self.max_len = max_len

    def __len__(self):
        return len(self.blocks)

    def __getitem__(self, i):
        b = self.blocks[i]
        ids = torch.tensor(b, dtype=torch.long)
        return {'input_ids': ids, 'attention_mask': torch.ones_like(ids),
                'labels': ids.clone()}


def train_lora(epochs, run_name, max_len=2048):
    """Train for `epochs`, saving a checkpoint each epoch (resumable)."""
    model, tok = build_trainable(max_len)
    ds = TokenBlocks(train_txt, tok, max_len)
    out = CKPT / run_name
    args = TrainingArguments(
        output_dir=str(out),
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=epochs,
        learning_rate=1e-4,
        lr_scheduler_type='linear',
        warmup_steps=400,
        weight_decay=0.0,
        optim='adamw_torch',
        bf16=True,
        gradient_checkpointing=True,
        logging_steps=10,
        save_strategy='epoch',
        save_total_limit=None,        # keep every epoch: Part C needs them
        report_to=[],
        seed=42,
    )
    trainer = Trainer(model=model, args=args, train_dataset=ds,
                      data_collator=default_data_collator)
    # resume automatically if a checkpoint exists, so a disconnect is cheap
    ckpts = sorted(out.glob('checkpoint-*')) if out.exists() else []
    trainer.train(resume_from_checkpoint=str(ckpts[-1]) if ckpts else None)
    model.save_pretrained(out / 'final')
    tok.save_pretrained(out / 'final')
    return out


In [ ]:
# Full recipe. ~1.5-3 h on an H100 for 64 MB / 10 epochs.
RUN_FULL = True
if RUN_FULL:
    adapter_dir = train_lora(epochs=10, run_name=f'{DATASET}_lora_r8_a32')
    print('adapter at', adapter_dir / 'final')


## Part C — training-quantity ablation (the experiment the paper omits)

This is the question the report needs answered: **is the fine-tuning gap a matter of training quantity, or of domain match?**

The paper contains no sweep over training amount, and its own log already hints the answer: eval loss is best at ~2 epochs and worsens through epoch 10. So we measure compression rate as a function of epochs, on the held-out test split, *and* on the MeDAL corpus whose domain the adapter was trained on.

Two readings are possible and they are distinguishable:

* if rate falls monotonically with epochs, the claim "more training helps" is supported and the gap is quantity;
* if rate bottoms out near epoch 2 and then rises, the adapter is overfitting and more training actively hurts — which would mean the gap is about *target-domain match and early stopping*, not quantity.


In [ ]:
ABLATION_EPOCHS = [1, 2, 5, 10]
ablation = {}
for ep in ABLATION_EPOCHS:
    d = train_lora(epochs=ep, run_name=f'{DATASET}_ablation_ep{ep}')
    rates = run_eval(str(d / 'final'), f'{DATASET}_ep{ep}', ctx_list=(8192,),
                     n_files=20)
    ablation[ep] = rates[8192]
    print(f'epochs={ep:>3}  compression rate = {rates[8192]:.4f} %')
    torch.cuda.empty_cache()

json.dump(ablation, open(RESULTS / f'ablation_epochs_{DATASET}.json', 'w'), indent=2)
print(json.dumps(ablation, indent=2))


### C2 — per-epoch checkpoints from a single 10-epoch run

Cheaper equivalent if Part B has already run: evaluate each saved epoch checkpoint instead of retraining. This isolates *quantity* with the training trajectory held fixed.


In [ ]:
import re
run_dir = CKPT / f'{DATASET}_lora_r8_a32'
per_epoch = {}
for ck in sorted(run_dir.glob('checkpoint-*'),
                 key=lambda p: int(re.search(r'(\d+)$', p.name).group(1))):
    step = int(re.search(r'(\d+)$', ck.name).group(1))
    rates = run_eval(str(ck), f'ep_ckpt{step}', ctx_list=(8192,), n_files=10)
    per_epoch[step] = rates[8192]
    print(f'step {step:>6}  rate = {rates[8192]:.4f} %')
    torch.cuda.empty_cache()
json.dump(per_epoch, open(RESULTS / f'per_epoch_{DATASET}.json', 'w'), indent=2)


## Part D — classical baselines on the same test files

Necessary control: without it, a compression rate says nothing. These use the paper's own settings (`zstd --ultra -22`, `brotli quality=11`, `lzma preset 9e` standing in for `7z -mx9`).


In [ ]:
import lzma, glob, zlib
import brotli
try:
    import zstandard as zstd
    HAVE_ZSTD = True
except ImportError:
    HAVE_ZSTD = False

files = sorted(glob.glob(str(DATA / 'test' / '*.csv')))[:N_TEST_FILES]
base = {}
for name, fn in [
    ('zlib 9', lambda b: zlib.compress(b, 9)),
    ('lzma preset 9e (7z-equivalent)', lambda b: lzma.compress(b, preset=9 | lzma.PRESET_EXTREME)),
    ('brotli q11', lambda b: brotli.compress(b, quality=11)),
] + ([('zstd --ultra -22', zstd.ZstdCompressor(level=22).compress)] if HAVE_ZSTD else []):
    tot_o = tot_c = 0
    for f in files:
        raw = open(f, 'rb').read()
        tot_o += len(raw); tot_c += len(fn(raw))
    base[name] = 100.0 * tot_c / tot_o
    print(f'{name:<34} rate={base[name]:7.3f}%  ratio={100/base[name]:6.3f}x')

json.dump(base, open(RESULTS / f'baselines_{DATASET}.json', 'w'), indent=2)


## Part E — package results

Produces a zip to download. Because `WORK` can be on Drive, re-running is safe: cursors are keyed by configuration, so finished files are skipped and a disconnect costs at most one window.


In [ ]:
import shutil
bundle = shutil.make_archive(str(WORK / f'lmcompress_results_{DATASET}'),
                             'zip', root_dir=str(RESULTS))
print('bundle:', bundle)
if USE_DRIVE:
    print('already on Drive:', WORK)
else:
    from google.colab import files
    files.download(bundle)


## What to send back

The zip contains:

| File | Meaning |
|---|---|
| `partA_raw_llama.json` | base model rate at each context length |
| `partA_shipped_lora.json` | the capsule adapter's rate |
| `ablation_epochs_*.json` | **the training-quantity curve** |
| `per_epoch_*.json` | same, per saved checkpoint |
| `baselines_*.json` | classical controls |
| `cursor_*.json` | resumable progress; also proof of what was actually coded |

**Expected outcome, so you can sanity-check it:** the base model should land near 9 % on MeDAL / 5.9 % on eurlex (the paper's LMCompress*), and the ablation curve should be *flat or U-shaped* rather than monotonically decreasing — matching the capsule's own training log, where eval loss bottomed out at ~2 epochs.
